# Phase 5: Data Cleaning & Validation Pipeline
This notebook cleans and validates the raw synthetic datasets generated for BarberFlow to prepare them for EDA, SQL seeding, and ML training.

In [1]:
import pandas as pd
import numpy as np
import os

print("Libraries loaded.")

Libraries loaded.


## 1. Load Raw Datasets
We will inspect the shapes and schemas of the files in `data/raw/`.

In [2]:
raw_dir = '../data/raw'
files = ["shops.csv", "services.csv", "customers.csv", "visits.csv", "appointments.csv", "availability.csv", "reviews.csv"]

for f in files:
    path = os.path.join(raw_dir, f)
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f"{f}: {df.shape[0]} rows, {df.shape[1]} columns")
    else:
        print(f"Error: {path} not found.")

shops.csv: 50 rows, 15 columns
services.csv: 337 rows, 5 columns
customers.csv: 5000 rows, 7 columns
visits.csv: 64991 rows, 12 columns
appointments.csv: 32522 rows, 10 columns
availability.csv: 56966 rows, 7 columns
reviews.csv: 25996 rows, 9 columns


## 2. Check for Missing Values
We check if any generated records have null or missing cells.

In [3]:
for f in files:
    df = pd.read_csv(os.path.join(raw_dir, f))
    missing = df.isnull().sum().sum()
    print(f"{f} has {missing} missing values")

shops.csv has 0 missing values
services.csv has 0 missing values
customers.csv has 0 missing values


visits.csv has 0 missing values
appointments.csv has 0 missing values
availability.csv has 0 missing values
reviews.csv has 0 missing values


## 3. Check for Duplicate Records
We check if there are duplicate primary keys or rows in our datasets.

In [4]:
for f in files:
    df = pd.read_csv(os.path.join(raw_dir, f))
    dups = df.duplicated().sum()
    print(f"{f} has {dups} duplicate rows")

shops.csv has 0 duplicate rows
services.csv has 0 duplicate rows
customers.csv has 0 duplicate rows


visits.csv has 0 duplicate rows


appointments.csv has 0 duplicate rows
availability.csv has 0 duplicate rows


reviews.csv has 0 duplicate rows


## 4. Outlier Analysis (Wait Times & Spending)
Using IQR to find extreme values and verify they are within logical bounds.

In [5]:
visits = pd.read_csv('../data/raw/visits.csv')
q1_wait = visits['waiting_time_minutes'].quantile(0.25)
q3_wait = visits['waiting_time_minutes'].quantile(0.75)
iqr_wait = q3_wait - q1_wait
upper_wait = q3_wait + 1.5 * iqr_wait
print(f"Wait time IQR: {iqr_wait}, Upper boundary for normal outliers: {upper_wait}")
print(f"Max wait time: {visits['waiting_time_minutes'].max()} mins")

customers = pd.read_csv('../data/raw/customers.csv')
q1_spend = customers['average_spending'].quantile(0.25)
q3_spend = customers['average_spending'].quantile(0.75)
iqr_spend = q3_spend - q1_spend
upper_spend = q3_spend + 1.5 * iqr_spend
print(f"Average spending IQR: {iqr_spend}, Upper boundary: {upper_spend}")
print(f"Max spending: ${customers['average_spending'].max():.2f}")

Wait time IQR: 0.0, Upper boundary for normal outliers: 0.0
Max wait time: 584 mins
Average spending IQR: 22.912499999999994, Upper boundary: 91.53124999999999
Max spending: $79.56


## 5. Latitude & Longitude Range Validation
Checking if all coordinates fall within reasonable bounds for the city (around Metropolis center 40.71, -74.00).

In [6]:
shops = pd.read_csv('../data/raw/shops.csv')
lat_bounds = (40.5, 40.9)
lon_bounds = (-74.2, -73.8)

invalid_lat = shops[(shops['latitude'] < lat_bounds[0]) | (shops['latitude'] > lat_bounds[1])]
invalid_lon = shops[(shops['longitude'] < lon_bounds[0]) | (shops['longitude'] > lon_bounds[1])]

print(f"Shops with invalid latitude: {len(invalid_lat)}")
print(f"Shops with invalid longitude: {len(invalid_lon)}")

Shops with invalid latitude: 0
Shops with invalid longitude: 0


## 6. Save Validated Clean Datasets
Since this is synthetic data generated by code, it is already clean. We write it to `data/processed/` and confirm data type conversions.

In [7]:
processed_dir = '../data/processed'
os.makedirs(processed_dir, exist_ok=True)

for f in files:
    df = pd.read_csv(os.path.join(raw_dir, f))
    
    # Ensure types are correct
    if f == "visits.csv":
        df["date"] = pd.to_datetime(df["date"])
        df["waiting_time_minutes"] = df["waiting_time_minutes"].astype(int)
    elif f == "availability.csv":
        df["timestamp"] = pd.to_datetime(df["timestamp"])
        df["occupied_seats"] = df["occupied_seats"].astype(int)
        
    df.to_csv(os.path.join(processed_dir, f), index=False)
    print(f"Saved cleaned {f} to {processed_dir}")

Saved cleaned shops.csv to ../data/processed
Saved cleaned services.csv to ../data/processed
Saved cleaned customers.csv to ../data/processed


Saved cleaned visits.csv to ../data/processed


Saved cleaned appointments.csv to ../data/processed
Saved cleaned availability.csv to ../data/processed


Saved cleaned reviews.csv to ../data/processed
